In [1]:
import os

os.listdir("/content")

['.config', 'Rd.zip', 'sample_data']

In [3]:
import zipfile

zip_path = "/content/Rd.zip"

extract_path = "/content/Rd"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully")

Dataset Extracted Successfully


In [7]:
import os

dataset_path = "/content/Rd/DATASET"

print(os.listdir(dataset_path)[:10])

['train', 'test']


In [8]:
import os

train_path = "/content/Rd/DATASET/train"

print(os.listdir(train_path))

['7', '3', '1', '4', '5', '2', '6']


In [ ]:
# ============================================
# IMPORTS
# ============================================

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import *
from tensorflow.keras.optimizers import Adam

# ============================================
# SETTINGS
# ============================================

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 35

dataset_path = "/content/Rd/DATASET/train"

# ============================================
# DATASET
# ============================================

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# ============================================
# CLASS NAMES
# ============================================

print(train_data.class_indices)

# ============================================
# MODEL
# ============================================

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(7, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# ============================================
# COMPILE
# ============================================

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ============================================
# CALLBACKS
# ============================================

checkpoint = ModelCheckpoint(
    "best_emotion_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

earlystop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2
)

callbacks = [checkpoint, earlystop, reduce_lr]

# ============================================
# TRAIN STAGE 1
# ============================================

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks
)

# ============================================
# FINE TUNING
# ============================================

base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

# ============================================
# SAVE MODEL
# ============================================

model.save("final_emotion_model.h5")

print("MODEL TRAINED SUCCESSFULLY")

Found 9819 images belonging to 7 classes.
Found 2452 images belonging to 7 classes.
{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
119/307 ━━━━━━━━━━━━━━━━━━━━ 1:17 411ms/step - accuracy: 0.2475 - loss: 2.3140